In [106]:
import pandas as pd
import numpy as np

path = "/Users/emiller/Documents/sf_tech/sf_datasets/"

# read in each tab as a dataframe
unc_results = pd.read_excel(path + "uncommon_schools_data_preparation_exercise.xlsx", sheet_name='AP Results')
unc_teacher = pd.read_excel(path + "uncommon_schools_data_preparation_exercise.xlsx", sheet_name='Teacher')
unc_roster = pd.read_excel(path + "uncommon_schools_data_preparation_exercise.xlsx", sheet_name='Student Roster')


print('complete')

complete


In [114]:
# format columns into lowercase, snake case
unc_roster.columns = unc_roster.columns.str.lower().str.replace(' ', '_')
unc_teacher.columns = unc_teacher.columns.str.lower().str.replace(' ', '_')
unc_results.columns = unc_results.columns.str.lower().str.replace(' ', '_')


# in teacher tab, school and subject columns appear to have their column names switched. data cleaning to proceed with updated (switched) column names
unc_teacher_renamed = unc_teacher.rename(columns={'school':'subject', 'subject':'school'})


In [ ]:
# inspect join keys to confirm formatting of values for merging data
# unc_results.subject.value_counts()
# unc_teacher_renamed.subject.value_counts()
# unc_teacher_renamed.school.value_counts()
# unc_results.subject.value_counts()
# unc_roster.school.value_counts()



In [108]:
# ######### merge data sets ########

# first, merge results dataset with roster dataset. second, merge output with teacher dataset.
merged_results_roster = unc_results.merge(unc_roster, how='left', left_on='student_number', right_on='student_id')
merged_all = merged_results_roster.merge(unc_teacher_renamed, how='left', on=['subject', 'school']) 

# drop the column 'student_id', as 'student_number' contains identical information. this datafrme is used for the final output
raw_output = merged_all.drop('student_id', axis='columns')




In [109]:

# ######## get count of students without an AP score ########
merged_roster_results = unc_roster.merge(unc_results, how='left', left_on='student_id', right_on='student_number')
students_no_score = merged_roster_results.student_number.isnull()    # 776 students out of 1110 had no AP scores


# ######## get list of schools without an AP score ########
full_school_list = list(unc_roster['school'].unique())
results_school_list = list(merged_results_roster['school'].unique())

scoreless_school_list = set(full_school_list) - set(results_school_list)
scoreless_school_list   # {'EVC', 'LBCHS'}


# ######## get list of teachers without an AP score ########
full_teacher_list = list(unc_teacher_renamed['teacher'].unique())
results_teacher_list = list(raw_output['teacher'].unique())

scoreless_teacher_list = set(full_teacher_list) - set(results_teacher_list)
scoreless_teacher_list      # empty list: all teachers had at least 1 student with an AP score

set()

In [115]:
########################### Data Cleaning/Observations ##############################
# ######## check row counts (run these lines separately to get output) ########
unc_results.shape           # 720 records in AP results tab
unc_teacher_renamed.shape   # 18 records in teacher tab
unc_roster.shape            # 1110 records in roster tab
raw_output.shape            # 720 records in output of merged data. this matches record count in results - success!

# ######## check data types (score should be a numeric field - success) ########
raw_output.dtypes



# ######## check raw_output for nulls (run these lines separately to get output) ########
raw_output.student_number.isnull().sum()      # 0, success!
raw_output.grade.isnull().sum()               # 0, success!
raw_output.score.isnull().sum()               # 0, success!
raw_output.teacher.isnull().sum()             # 0, success!
missing_teacher_name = raw_output[raw_output['teacher'].isnull()]    
# appears to be missing a teacher: English Lang and Composition, 12th Grade, NDHS




# ######## check for duplicate (or partial duplicate) records in output ########
# partial duplicates: combination of student_number + subject represent a 'unique' record
# full duplicates: combination of student_number + subject + score represent a 'unique' record 

# duplicate check (full or partial)
output_dupe_no_score = raw_output.groupby(['student_number','subject'], as_index=False).size()
dupe_no_score = output_dupe_no_score[output_dupe_no_score['size'] > 1]      
dupe_no_score_list = dupe_no_score['student_number'].to_list()
# there are 80 instances where a student has multiple scores for a subject


# duplicate check - full duplicates only
output_dupe_with_score = raw_output.groupby(['student_number','subject', 'score'], as_index=False).size()
dupe_with_score = output_dupe_with_score[output_dupe_with_score['size'] > 1]      
dupe_with_score_list = dupe_with_score['student_number'].to_list()
# there are 78 instances where a student has multiple records with the same score for a subject 


# duplicate check - partial duplicates only
partial_dupe = set(dupe_no_score_list) - set(dupe_with_score_list)
# There are 2 students who have multiple different scores for the same exam (204899629, 206316473). Both are from REH, took the World History exam, and have Oktani as teacher



In [111]:

# ######## create 2 new fields: graduating_cohort_year and passed_exam. 
conditions = [
    (raw_output['grade'] == '12th Grade')
    , (raw_output['grade'] == '11th Grade')
    , (raw_output['grade'] == '10th Grade')
    , (raw_output['grade'] == '9th Grade')
]
choices = [2022, 2023, 2024, 2025]
raw_output['graduating_cohort_year'] = np.select(conditions, choices)

raw_output['passed_exam'] = raw_output['score'] >= 3


In [112]:
final_output = raw_output[['student_number', 'score', 'subject', 'school', 'graduating_cohort_year', 'passed_exam', 'lunch_status', 'ethnicity', 'gender', 'teacher']]

new_column_names = ['Student Number', 'Score', 'Subject', 'High School', 'Graduating cohort year', 'Passed Exam', 'FRPL Status', 'Ethnicity', 'Gender', 'Teacher Name']
final_output.columns = new_column_names


In [ ]:

final_output.to_excel(path + "Miller_E_Sr_BI_Developer_data_output.xlsx", index=False)
